# 30 · Phase 4: Court Judgment Prediction (binary, macro-F1) on top of the Phase-1 models
Judgments are long (median ~2.9k words; train max 412k), so each document is reduced to `max_chunks` windows of
`chunk_len` tokens (`strategy`: `last` | `first_last` | `spread`); chunk states are pooled (`token_pooling`) and chunk
vectors aggregated (`chunk_agg`: `mean` | `max` | `attention`) -- the long-document / hierarchical comparison of §10.1.
Selection on the official dev split. Runs go to `MyDrive/cs781/cjpe_runs/`. The `label` field in test.json is never read.

Data facts to plan with: train 37,387 docs, **59% label 0 / 41% label 1**, dev/test balanced -> consider
`class_weighting='balanced'` / `tune_threshold=True`. Train states the outcome explicitly near the end in 17.6% of docs vs
8.2% in dev/test -> `drop_outcome=True` removes those sentences (a shortcut the model could otherwise learn).


Brief (§10.1–10.2): **all four** Phase-1 models; ≥4 different experiments; **LoRMA compulsory**; **1 knowledge-distillation**
run (`kd_teacher=`, same strategy/chunking as the teacher).

In [ ]:
import os; os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# ---- paths + locate the code (lm_lib/ and models/) ----
# Put the code on Drive in ONE of these ways:
#   (a) upload the folders lm_lib/ and models/ (+ hf_token.txt) into MyDrive/cs781_capstone/   [or any folder]
#   (b) upload cs781_code.zip (made by make_code_zip.py) anywhere in MyDrive -> it is unzipped automatically
from google.colab import drive
drive.mount('/content/drive')
import sys, os, glob, zipfile, importlib, shutil, time
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; must precede CUDA init
MYDRIVE    = '/content/drive/MyDrive'
DRIVE_ROOT = f'{MYDRIVE}/cs781'            # data + runs live here
DRIVE_DATA = f'{DRIVE_ROOT}/data'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
LOCAL_DATA = '/content/data'

def find_code_dir():
    levels = ['', '*/', '*/*/', '*/*/*/']
    # 1) a folder on Drive containing lm_lib/ (newest wins if there are several copies)
    hits = [h for lv in levels for h in glob.glob(f'{MYDRIVE}/{lv}lm_lib/trainer.py')]
    if hits:
        return os.path.dirname(os.path.dirname(max(hits, key=os.path.getmtime)))
    # 2) the newest cs781_code*.zip on Drive -> always re-extracted fresh (so an updated zip replaces old code)
    zips = [z for lv in levels for z in glob.glob(f'{MYDRIVE}/{lv}cs781_code*.zip')]
    if zips:
        z = max(zips, key=os.path.getmtime)
        shutil.rmtree('/content/cs781_code', ignore_errors=True)
        with zipfile.ZipFile(z) as f:
            f.extractall('/content/cs781_code')
        print('unzipped', z, '| zip modified', time.ctime(os.path.getmtime(z)))
        return '/content/cs781_code'
    raise FileNotFoundError('Could not find lm_lib/ on Drive. Upload the lm_lib/ and models/ folders '
                            '(e.g. to MyDrive/cs781_capstone/) or upload cs781_code.zip to MyDrive, then re-run.')

CODE_DIR = find_code_dir()
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
importlib.invalidate_caches()
# if lm_lib/models were imported earlier in this kernel, drop them so the fresh code is used
for _m in [m for m in list(sys.modules) if m == 'lm_lib' or m.startswith(('lm_lib.', 'models.')) or m == 'models']:
    del sys.modules[_m]
print('CODE_DIR =', CODE_DIR, '| contents:', sorted(os.listdir(CODE_DIR)))
os.makedirs(DRIVE_DATA, exist_ok=True); os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
!pip -q install ijson "huggingface_hub[hf_xet]" tokenizers
import os
from huggingface_hub import login, whoami

def get_hf_token():
    if os.environ.get('HF_TOKEN'):
        return os.environ['HF_TOKEN'].strip()
    p = f'{CODE_DIR}/hf_token.txt'
    if os.path.exists(p):
        return open(p).read().strip()
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception as e:
        raise RuntimeError(f'No HF token found: set $HF_TOKEN or create {p}') from e

HF_TOKEN = get_hf_token()
login(token=HF_TOKEN, add_to_git_credential=False)
print('logged in to Hugging Face as', whoami(token=HF_TOKEN)['name'])

In [ ]:
from huggingface_hub import hf_hub_download
import shutil
P4 = '/content/p4'; os.makedirs(P4, exist_ok=True)
for f in ['Train/train.json', 'Train/dev.json', 'Test/test.json']:
    p = hf_hub_download('Exploration-Lab/CS-781-Capstone', f'Phase_4_CJPE/{f}', repo_type='dataset', token=HF_TOKEN)
    shutil.copyfile(p, f'{P4}/{os.path.basename(f)}')
!ls -lh {P4}

In [ ]:
PHASE1 = {
    'A': f'{RUNS_DIR}/A_qwen_dense_full',
    'B': f'{RUNS_DIR}/B_encdec',
    'C': f'{RUNS_DIR}/C_gdn_hybrid',
    'D': f'{RUNS_DIR}/D_mla_moe',
}
for k, v in PHASE1.items():
    print(k, v, '| ckpts:', sorted(f for f in os.listdir(v) if f.endswith('.pt')) if os.path.isdir(v) else 'MISSING')
if os.path.isdir(PHASE1['C']):
    !pip -q install flash-linear-attention
# after any pip install, restart the kernel once (os.kill(os.getpid(), 9)) before training -- see the sympy issue

In [ ]:
from lm_lib.cjpe.trainer import CJPEConfig, train_cjpe, predict_test
cfg = CJPEConfig(
    phase1_run=PHASE1['B'], phase1_ckpt='best',
    method='full', method_kwargs={},                # student: B's encoder, fully fine-tuned
    data_dir=P4, out_dir=f'{DRIVE_ROOT}/cjpe_runs',
    run_name='B_encdec__kd_from_A__last4x512',      # explicit, so it doesn't overwrite the LoRMA run on B
    strategy='last', chunk_len=512, max_chunks=4,   # must match the teacher exactly
    token_pooling='mean', chunk_agg='mean',
    normalize_text=True, drop_outcome=False,        # same as the teacher
    n_train=0,
    batch_size=8, grad_accum=1, grad_checkpoint=False,
    epochs=2, lr=5e-5, head_lr=1e-3, weight_decay=0.01, warmup_frac=0.1, dropout=0.1,
    class_weighting='balanced',
    tune_threshold=False,
    eval_every=1500,
    kd_teacher=f'{DRIVE_ROOT}/cjpe_runs/A_qwen_dense_full__full__last4x512',
    kd_alpha=0.5,                  # weight of the teacher's s
    kd_temperature=2.0,
    stop_at='',
)
print(cfg.run_name)

In [ ]:
results = train_cjpe(cfg)

In [ ]:
# test.json -> prediction.json [{"id", "pred"}] + submission.zip (uses best.pt and its threshold)
run_dir = f'{DRIVE_ROOT}/cjpe_runs/{cfg.run_name}'
zip_path, test_preds = predict_test(run_dir)
print('submission on Drive:', zip_path)

In [ ]:
import json, glob, pandas as pd
rows = [json.load(open(p)) for p in glob.glob(f'{DRIVE_ROOT}/cjpe_runs/*/results.json')]
pd.DataFrame(rows)[['run', 'phase1_run', 'method', 'strategy', 'max_chunks', 'chunk_len', 'chunk_agg', 'drop_outcome', 'n_train',
                    'kd_teacher', 'best_dev_macro_f1', 'threshold', 'trainable_pct']] \
  .sort_values('best_dev_macro_f1', ascending=False)